# Fraud Detection: Re-tuning LightGBM on Feature Set v2

> **Team decision (2026-10-22):** the final model uses the notebook 11 settings. This notebook documents the re-tuning attempt and why we did not use it. It is not part of the presentation.

Milestone 3. In [13_feature-selection.ipynb](13_feature-selection.ipynb) the team adopted **feature set v2**: time features and meter features added, the 4 level max columns removed, **30 features** in total (`models/selected_features.json`, decision in [protocol.md](protocol.md)). With the settings tuned in [11_hyperparameter-tuning.ipynb](11_hyperparameter-tuning.ipynb), the CV ROC-AUC rose from 0.769 to 0.848.

Those settings were tuned for the old 24 features. Because the features changed a lot, we **tune again** with exactly the same method as notebook 11:
- random search with 30 combinations and the same 8 ranges,
- the same 5 stratified folds on the training split, scored with **ROC-AUC** (PR-AUC next to it),
- the fraud class stays weighted (`class_weight="balanced"`),
- the validation split is used **once**, at the end.

The notebook shows only tables and numbers, no charts.

In [1]:
# libraries for data handling, the model, the random search and the metrics
import json
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate

pd.set_option("display.max_columns", 40)
FEATURE_DIR = Path("data/features")
MODEL_DIR = Path("models")
RANDOM_STATE = 42

## 1. Load feature set v2

The `_v2` tables from notebook 13 (same clients and the same 80/20 split as before), reduced to the 30 chosen columns. For the comparison at the end we also load the old 24-feature tables.

In [2]:
# read one feature table from data/features, with client_id as the index
def load(name):
    return pd.read_csv(FEATURE_DIR / f"{name}.csv", index_col="client_id")

# the 30 columns chosen in notebook 13
selected = json.loads((MODEL_DIR / "selected_features.json").read_text())["features"]

X_train, X_valid = load("X_train_v2")[selected], load("X_valid_v2")[selected]
X_train_old, X_valid_old = load("X_train"), load("X_valid")
y_train, y_valid = load("y_train")["target"], load("y_valid")["target"]

# features and labels must belong to the same clients, in the same order
assert (X_train.index == y_train.index).all() and (X_valid.index == y_valid.index).all()
print("train:", X_train.shape, "| valid:", X_valid.shape)

train: (108394, 30) | valid: (27099, 30)


## 2. Scoring helper

The same helper as in notebooks 09 to 13 (precision, recall and F1 at the F1-best threshold are an upper bound).

In [3]:
def score(y_true, proba):
    # precision and recall for every possible threshold
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # F1 for every threshold; the small number avoids dividing by zero
    f1_curve = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    # the F1-best threshold: the cut-off with the highest F1 on this data
    best = f1_curve.argmax()
    return {
        "valid ROC-AUC": roc_auc_score(y_true, proba),
        "valid PR-AUC": average_precision_score(y_true, proba),
        "precision@best": precision[best],
        "recall@best": recall[best],
        "F1@best": f1_curve[best],
    }

## 3. Reference: the settings from notebook 11 on the 30 features

The score to beat. It should match the "combined set" in notebook 13 (CV ROC-AUC 0.848).

In [4]:
# the same 5 stratified folds as in notebooks 10 to 13
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# the settings tuned in notebook 11 (class weight "balanced" included)
old_params = json.loads((MODEL_DIR / "lightgbm_best_params.json").read_text())
old_model = LGBMClassifier(**old_params, n_jobs=-1, verbose=-1)
old_cv = cross_validate(old_model, X_train, y_train, cv=cv,
                        scoring=["roc_auc", "average_precision"], return_train_score=True)

print(f"old settings, CV ROC-AUC: {old_cv['test_roc_auc'].mean():.4f} (± {old_cv['test_roc_auc'].std():.4f})")
print(f"old settings, CV PR-AUC:  {old_cv['test_average_precision'].mean():.4f}")

old settings, CV ROC-AUC: 0.8476 (± 0.0058)
old settings, CV PR-AUC:  0.3048


## 4. Random search

Exactly the same search as in notebook 11: 30 random combinations × 5 folds = 150 fits. See notebook 11 for what each setting does. The class weight is fixed (team decision), not searched.

In [5]:
# the same ranges as in notebook 11
param_distributions = {
    "n_estimators": randint(100, 1001),
    "learning_rate": uniform(0.01, 0.14),        # 0.01 to 0.15
    "num_leaves": randint(15, 128),
    "max_depth": [-1, 4, 6, 8],
    "min_child_samples": randint(20, 301),
    "subsample": uniform(0.6, 0.4),              # 0.6 to 1.0
    "colsample_bytree": uniform(0.5, 0.5),       # 0.5 to 1.0
    "reg_lambda": loguniform(1e-3, 10),
}

search = RandomizedSearchCV(
    LGBMClassifier(class_weight="balanced", subsample_freq=1, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    param_distributions,
    n_iter=30,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
    refit="roc_auc",                # choose and refit by ROC-AUC, our main metric
    cv=cv,
    return_train_score=True,        # needed for the overfitting check
    random_state=RANDOM_STATE,
    verbose=1,
)
# fit on the training split only; the validation split is not touched here
search.fit(X_train, y_train)

print(f"best CV ROC-AUC: {search.best_score_:.4f}")
{k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in search.best_params_.items()}

Fitting 5 folds for each of 30 candidates, totalling 150 fits


best CV ROC-AUC: 0.8477


{'colsample_bytree': 0.6622,
 'learning_rate': 0.0271,
 'max_depth': -1,
 'min_child_samples': 250,
 'n_estimators': 289,
 'num_leaves': 111,
 'reg_lambda': 0.0123,
 'subsample': 0.8591}

In [6]:
# the 5 best combinations: is the top stable, or one lucky draw?
results = pd.DataFrame(search.cv_results_)
results["gap"] = results["mean_train_roc_auc"] - results["mean_test_roc_auc"]
columns = ["mean_test_roc_auc", "std_test_roc_auc", "mean_test_pr_auc", "gap"] + [c for c in results.columns if c.startswith("param_")]
results.sort_values("mean_test_roc_auc", ascending=False)[columns].head(5).round(4)

,mean_test_roc_auc,std_test_roc_auc,mean_test_pr_auc,gap,param_colsample_bytree,param_learning_rate,param_max_depth,param_min_child_samples,param_n_estimators,param_num_leaves,param_reg_lambda,param_subsample
19,0.8477,0.0057,0.3104,0.1117,0.6622,0.0271,-1,250,289,111,0.0123,0.8591
28,0.8476,0.0058,0.3048,0.0686,0.8676,0.0393,6,118,362,30,1.8573,0.8221
4,0.8473,0.0056,0.3016,0.0829,0.8926,0.0380,6,263,931,17,2.7527,0.8721
23,0.8469,0.0049,0.2925,0.0563,0.9883,0.0675,6,243,324,15,0.0091,0.6580
26,0.8465,0.0057,0.2957,0.0709,0.9177,0.0549,4,278,759,73,0.0076,0.8845


## 5. Old vs re-tuned settings (cross-validation)

Both on the 30 features and the same folds.

In [7]:
best = results.loc[search.best_index_]
cv_comparison = pd.DataFrame({
    "old settings (notebook 11)": {
        "CV ROC-AUC": old_cv["test_roc_auc"].mean(),
        "CV ROC-AUC std": old_cv["test_roc_auc"].std(),
        "CV PR-AUC": old_cv["test_average_precision"].mean(),
        "train ROC-AUC": old_cv["train_roc_auc"].mean(),
    },
    "re-tuned settings": {
        "CV ROC-AUC": best["mean_test_roc_auc"],
        "CV ROC-AUC std": best["std_test_roc_auc"],
        "CV PR-AUC": best["mean_test_pr_auc"],
        "train ROC-AUC": best["mean_train_roc_auc"],
    },
}).T
cv_comparison["gap"] = cv_comparison["train ROC-AUC"] - cv_comparison["CV ROC-AUC"]

gain = cv_comparison.iloc[1]["CV ROC-AUC"] - cv_comparison.iloc[0]["CV ROC-AUC"]
print(f"improvement in CV ROC-AUC from re-tuning: {gain:+.4f}")
cv_comparison.round(4)

improvement in CV ROC-AUC from re-tuning: +0.0001


,CV ROC-AUC,CV ROC-AUC std,CV PR-AUC,train ROC-AUC,gap
old settings (notebook 11),0.8476,0.0058,0.3048,0.9162,0.0686
re-tuned settings,0.8477,0.0057,0.3104,0.9594,0.1117


## 6. Final check on the validation split (used once)

Three models, all fitted on the whole training split and scored on the validation clients:
1. the **first model**: 24 features, settings from notebook 11 (the model of the error analysis),
2. **30 features with the old settings**,
3. **30 features with the re-tuned settings**.

In [8]:
# 1. first model: old 24 features and old settings
first_model = LGBMClassifier(**old_params, n_jobs=-1, verbose=-1).fit(X_train_old, y_train)
# 2. new features, old settings
old_model.fit(X_train, y_train)
# 3. new features, re-tuned settings (already refitted on the whole training split by the search)
tuned_model = search.best_estimator_

valid_comparison = pd.DataFrame({
    "first model (24 features)": score(y_valid, first_model.predict_proba(X_valid_old)[:, 1]),
    "30 features, old settings": score(y_valid, old_model.predict_proba(X_valid)[:, 1]),
    "30 features, re-tuned": score(y_valid, tuned_model.predict_proba(X_valid)[:, 1]),
}).T
valid_comparison.round(4)

,valid ROC-AUC,valid PR-AUC,precision@best,recall@best,F1@best
first model (24 features),0.7742,0.1863,0.2170,0.3034,0.2530
"30 features, old settings",0.8585,0.3195,0.3559,0.4038,0.3783
"30 features, re-tuned",0.8592,0.3278,0.3183,0.4719,0.3802


## 7. Save the re-tuned model

The model and its settings are saved next to the first model, so the final notebook can use them.

In [9]:
MODEL_DIR.mkdir(exist_ok=True)

# the trained trees in LightGBM's own text format
tuned_model.booster_.save_model(str(MODEL_DIR / "lightgbm_v2_tuned.txt"))

# the chosen settings, including the fixed ones, so the model can be rebuilt exactly
params = {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in search.best_params_.items()}
params.update({"class_weight": "balanced", "subsample_freq": 1, "random_state": RANDOM_STATE})
(MODEL_DIR / "lightgbm_v2_best_params.json").write_text(json.dumps(params, indent=2))

print(sorted(p.name for p in MODEL_DIR.glob("lightgbm_v2_*")))

['lightgbm_v2_best_params.json', 'lightgbm_v2_tuned.txt']


## Result

> **Team decision (2026-10-22):** the final model uses the notebook 11 settings. This notebook documents the re-tuning attempt and why we did not use it. It is not part of the presentation.

| Model (validation split) | Features | CV ROC-AUC | Gap | Valid ROC-AUC | Valid PR-AUC |
|---|---|---|---|---|---|
| First model (notebook 11) | 24 | 0.769 | 0.093 | 0.774 | 0.186 |
| **30 features, old settings** | 30 | **0.848** | **0.069** | 0.859 | 0.320 |
| 30 features, re-tuned | 30 | 0.848 | 0.112 | 0.859 | 0.328 |

- **Re-tuning brings nothing.** The best combination improves the CV ROC-AUC by only +0.0001, and the validation ROC-AUC by +0.0007. Both are far below the spread between the folds (about 0.006). The settings from notebook 11 are already as good on the new features.
- **The re-tuned model overfits more.** It uses deeper trees (111 leaves, no depth limit), and its train-CV gap rises from 0.069 to 0.112. It scores the same on new clients, so the extra complexity buys nothing.
- **The top of the search is flat again**: the 5 best combinations all score between 0.8465 and 0.8477.
- The big step came from the **features**, not from the settings: +0.079 CV ROC-AUC (notebook 13) vs +0.0001 here.

**Final model (team decision):** **30 features with the settings from notebook 11** (`models/lightgbm_best_params.json`). It scores the same as the re-tuned model and overfits less. The re-tuned model stays saved as `models/lightgbm_v2_tuned.txt` as a record of this attempt; it is not used.

**Next step (notebook 15):** refit the final model on all labelled clients (train plus validation) and predict the 58,069 Zindi test clients.